In [1]:
# Run this only once if packages are missing:
# %pip install -q fasttext-wheel langdetect pandas
print('Using existing environment packages.')

Using existing environment packages.


## Language detection from lyrics (FastText)

Reads `lyrics.csv`, detects the language of each song using the **first 200 characters** of its lyrics
via FastText's pre-trained language identification model (`lid.176.ftz`),
and writes `lyrics_lang.csv` to the same directory.

Output: all original columns from `lyrics.csv` + a new `language` column.

In [5]:
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import fasttext
import fasttext.FastText

# Idempotent monkey-patch: fix np.array(copy=False) error in NumPy 2.x
if not hasattr(fasttext.FastText._FastText, '_orig_predict'):
    fasttext.FastText._FastText._orig_predict = fasttext.FastText._FastText.predict

    def _safe_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
        import numpy as np
        _old_array = np.array
        def _array_compat(*args, **kwargs):
            kwargs.pop('copy', None)
            return _old_array(*args, **kwargs)
        np.array = _array_compat
        try:
            return fasttext.FastText._FastText._orig_predict(self, text, k=k, threshold=threshold, on_unicode_error=on_unicode_error)
        finally:
            np.array = _old_array

    fasttext.FastText._FastText.predict = _safe_predict
    print('NumPy 2.x patch applied.')
else:
    print('Patch already applied, skipping.')

# Resolve project root whether notebook runs from workspace root or notebooks/
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# ── Download FastText lid model if not cached ────────────────────────────────
MODEL_URL = 'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz'
MODEL_PATH = ROOT / 'models' / 'lid.176.ftz'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print(f'Downloading FastText lid model \u2192 {MODEL_PATH} ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Done.')
else:
    print(f'Model already cached at {MODEL_PATH}')

ft_model = fasttext.load_model(str(MODEL_PATH))
print('FastText model loaded.')

Patch already applied, skipping.
Model already cached at /Users/wednesday/Documents/GitHub/lyrics_analysis/models/lid.176.ftz
FastText model loaded.


In [3]:
DATE_PATH = '2026/03/05'
LYRICS_DIR = ROOT / 'data' / 'processed' / 'lyrics' / DATE_PATH
LYRICS_IN  = LYRICS_DIR / 'lyrics.csv'
LANG_OUT   = LYRICS_DIR / 'lyrics_lang.csv'

df = pd.read_csv(LYRICS_IN)
print(f'Loaded {len(df)} songs from {LYRICS_IN}')

def make_snippet(lyrics: str, n_chars: int = 200) -> str:
    if not isinstance(lyrics, str) or not lyrics.strip():
        return ''
    return lyrics[:n_chars].replace('\n', ' ').strip()

snippets = df['lyrics'].map(make_snippet)
valid_mask = snippets.ne('')
langs = pd.Series('unknown', index=df.index, dtype='object')

# FastText supports list input; this is much faster than row-wise apply.
if valid_mask.any():
    labels, _scores = ft_model.predict(snippets[valid_mask].tolist(), k=1)
    langs.loc[valid_mask] = [lbls[0].replace('__label__', '') for lbls in labels]

lang_df = df.copy()
lang_df['language'] = langs
lang_df.to_csv(LANG_OUT, index=False)

print(f'Saved {len(lang_df)} rows \u2192 {LANG_OUT}')
print('\nLanguage distribution:')
print(lang_df['language'].value_counts().to_string())
display(lang_df.head(10))


Loaded 754 songs from /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv
Saved 754 rows → /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang.csv

Language distribution:
language
en         352
es         212
zh          64
unknown     63
ko          24
tr           8
pt           6
it           4
ru           3
id           3
ja           3
de           3
gd           2
ar           2
fr           2
he           1
vi           1
pl           1


,rank,artist,title,region,spotify_uri,lyrics,language
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,"(Dímelo, ¿me vas a dar lo que yo pido?)\nDame ...",es
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,"ARIA VEGA, Ryan Castro\nLa costeñita premium y...",es
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",es
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,"Yeah, yeah\nYeah, yeah\n\nMi amor, culeemos co...",es
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,O-O-Ovy On The Drums\nYou're the apple of my e...,en
5,6,Beéle,no tiene sentido,Colombia,1HEwEN64NjgTaHmo7LfkX8,"(¡Uh-wu-wu-wu-wu-wu-wu-wu!)\nBaby, ¿Qué tú esp...",es
6,7,"J Balvin, Ryan Castro, DJ Snake",Tonto,Colombia,7mU1fei7P9h4mpjP2Otdw5,No se puede decir grosería'\nSoy Godzilla y pr...,es
7,8,Beéle,quédate,Colombia,6VfL3MEuYeJbDlD8m011HR,¿Cómo le explico al corazón que quieres irte?\...,es
8,9,Grupo Firme,El Beneficio De La Duda,Colombia,5yXt80BNZGbmHFd0NHZHNn,Sé que ahora lo último que quieres es saber có...,es
9,10,"Yeison Jimenez, Luis Alfonso",Destino Final,Colombia,2E4TYekUduml1DWIqQWNcj,"Y cántele lindo, ¡señorazo!\n\nYo sé que algún...",es


In [9]:
# check lyrics after tranlsatoin
DATE_PATH = '2026/03/05'
LYRICS_DIR = ROOT / 'data' / 'processed' / 'lyrics' / DATE_PATH
LYRICS_IN  = LYRICS_DIR / 'lyrics_lang_trans.csv'
LANG_OUT   = LYRICS_DIR / 'lyrics_lang_trans2.csv'

df = pd.read_csv(LYRICS_IN)
print(f'Loaded {len(df)} songs from {LYRICS_IN}')

def make_snippet(lyrics: str, n_chars: int = 200) -> str:
    if not isinstance(lyrics, str) or not lyrics.strip():
        return ''
    return lyrics[:n_chars].replace('\n', ' ').strip()

snippets = df['lyrics_en'].map(make_snippet)
valid_mask = snippets.ne('')
langs = pd.Series('unknown', index=df.index, dtype='object')

# FastText supports list input; this is much faster than row-wise apply.
if valid_mask.any():
    labels, _scores = ft_model.predict(snippets[valid_mask].tolist(), k=1)
    langs.loc[valid_mask] = [lbls[0].replace('__label__', '') for lbls in labels]

lang_df = df.copy()
lang_df['language'] = langs
lang_df.to_csv(LANG_OUT, index=False)

print(f'Saved {len(lang_df)} rows \u2192 {LANG_OUT}')
print('\nLanguage distribution:')
print(lang_df['language'].value_counts().to_string())
display(lang_df.head(10))


Loaded 754 songs from /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans.csv
Saved 754 rows → /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics_lang_trans2.csv

Language distribution:
language
unknown    415
en         336
it           2
pl           1


,rank,artist,title,region,spotify_uri,lyrics,language,lyrics_en
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,"(Dímelo, ¿me vas a dar lo que yo pido?)\nDame ...",en,"(Tell me, are you going to give me what I ask ..."
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,"ARIA VEGA, Ryan Castro\nLa costeñita premium y...",en,"ARIA VEGA, Ryan Castro\nThe premium costñita a..."
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",en,"Kapo, Ryan Castro, Gangsta\nWhat a joke, SOG\n..."
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,"Yeah, yeah\nYeah, yeah\n\nMi amor, culeemos co...",en,"Yeah, yeah\nYeah, yeah\n\nMy love, let's fuck ..."
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,O-O-Ovy On The Drums\nYou're the apple of my e...,unknown,NaN
5,6,Beéle,no tiene sentido,Colombia,1HEwEN64NjgTaHmo7LfkX8,"(¡Uh-wu-wu-wu-wu-wu-wu-wu!)\nBaby, ¿Qué tú esp...",en,"(Uh-wu-wu-wu-wu-wu-wu-wu!)\nBaby, what are you..."
6,7,"J Balvin, Ryan Castro, DJ Snake",Tonto,Colombia,7mU1fei7P9h4mpjP2Otdw5,No se puede decir grosería'\nSoy Godzilla y pr...,en,You can't say rude things\nI am Godzilla and I...
7,8,Beéle,quédate,Colombia,6VfL3MEuYeJbDlD8m011HR,¿Cómo le explico al corazón que quieres irte?\...,en,How do I explain to my heart that you want to ...
8,9,Grupo Firme,El Beneficio De La Duda,Colombia,5yXt80BNZGbmHFd0NHZHNn,Sé que ahora lo último que quieres es saber có...,en,I know the last thing you want now is to know ...
9,10,"Yeison Jimenez, Luis Alfonso",Destino Final,Colombia,2E4TYekUduml1DWIqQWNcj,"Y cántele lindo, ¡señorazo!\n\nYo sé que algún...",en,"And sing to him beautifully, sir!\n\nI know th..."
